# 0. IMPORTAR LIBRERIAS

In [1]:
#básicos
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import numpy as np
import numpy.linalg as la
import math # to use isnan
from time import perf_counter as clock

#grafos
import networkx as nx


# de detección de anomalías
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from pyod.models.kde import KDE
from  pyod.models.sampling import Sampling
from pyod.models.gmm import GMM
# from pyod.models.lunar import LUNAR
from pyod.models.ecod import ECOD


# de minería de asociación
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori,fpgrowth,association_rules,fpmax
from scipy.spatial import ConvexHull
from sklearn.preprocessing import KBinsDiscretizer

%matplotlib inline

In [2]:
sns.set_theme()
pd.set_option("display.max_rows", None, "display.max_columns", None)

# 1. CARGAR DATASET

In [3]:
dfs=pd.read_csv('dfs_fase3COMPLETO.csv')
dfs['indice']=dfs.index 

In [4]:
dfs = dfs.rename(columns={
    'SE1_pred': 'SE1',
    'SE2_pred': 'SE2',
    'SE3_pred': 'SE3',
    "cluster_ganador": "clase"
})

dfs.head()

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice
0,Transporte,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,Si,4,182,2882,11756400,-0.230632,-0.122036,-0.001522,-0.002303,0.134220,0.176406,-0.056739,-0.030522,-0.149470,0.164997,-0.101998,-0.269728,0.280453,0.097403,0.188180,-0.512083,0.142109,0.154056,-0.289179,0.131029,0.087097,0.158542,0.065064,0.098843,0.241269,-0.301444,-0.155364,-0.045086,0.110323,-0.290308,-0.021428,-0.183431,-0.165765,0.050169,0.516933,-0.699410,-0.640868,-1.197668,0,0
1,agricultura,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,0,242,2812,48000000,-0.502233,-0.314990,0.007575,0.003116,0.000818,0.132630,0.047187,-0.261014,-0.207582,0.041610,0.177906,0.090063,0.021362,0.113700,-0.266953,0.032131,0.100277,0.052154,0.042238,-0.037100,0.025141,-0.161671,-0.109760,0.206130,0.286944,-0.508839,-0.089393,0.386624,0.709633,0.170777,0.529664,-0.574655,-0.339869,0.541785,0.226969,1.292304,1.291991,-0.388060,2,1
2,Salud y Protección Social,Ejecutivo,Centralizada,Prestación de servicios,Contratación régimen especial,No,No,No,No,No,No,No Válido,No,Si,0,132,2856,8074779,1.089181,-0.408082,0.010449,0.011622,0.011589,0.238742,-0.031481,-0.343014,-0.053873,-0.060902,-0.056533,-0.014609,0.334820,0.020835,0.121477,-0.550172,0.169496,-0.077804,-0.090625,0.054882,0.036406,0.091204,0.087001,-0.132702,0.073605,0.015176,-0.013934,0.034000,0.075565,0.016952,0.002795,0.079164,-0.010538,0.088693,-0.077400,1.541353,1.246222,-0.408194,2,2
3,Educación Nacional,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,3,15,3035,14885150,-0.338042,-0.278063,0.010924,0.013671,-0.009648,-0.363740,-0.090634,0.265894,0.020466,-0.003160,-0.173324,0.609988,-0.310310,-0.457564,-0.354598,-0.334695,-0.015035,0.882740,0.052598,-0.168797,-0.091880,0.185196,-0.353456,-0.112787,0.114576,-0.231244,0.083896,0.326041,-0.051403,0.082964,-0.121959,-0.043341,-0.058637,0.063921,0.094111,-0.332191,-0.062011,-0.105104,0,3
4,Inclusión Social y Reconciliación,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,21,305,2888,28820000,-0.487375,-0.318814,0.011714,0.008560,0.014443,0.285101,-0.016738,-0.257863,-0.212083,0.107955,0.124422,-0.003640,-0.141943,0.141153,-0.214277,0.193188,0.021567,-0.002867,-0.011424,-0.052882,0.066520,-0.447140,0.143064,0.269374,0.558160,-0.470381,0.162990,-0.043987,-0.366043,1.742416,-0.265758,-0.090741,0.681484,0.093808,-0.622647,-0.693104,0.502032,0.551954,0,4


# 2. DETECCIÓN DE ANOMALÍAS

## 2.1 LOCAL OUTLIER FACTOR

In [114]:
nolineales=['SE1','SE2','SE3']
dfss=dfs.sample(8000,random_state=42).reset_index(drop=True)
datos=dfss.loc[:,nolineales]
estimator=LocalOutlierFactor(n_neighbors=10,novelty=True,contamination=0.01)
X=datos.values
estimator.fit(X)
dfss['outlier']=estimator.predict(X)

fig = px.scatter_3d(dfss, x='SE1', y='SE2', z='SE3',color='outlier',opacity=0.7,hover_data=['clase','indice'])
scene=dict(aspectratio=dict(x=1, y=1,z=1))
fig.update_traces(marker_size = 2)
fig.update_scenes(aspectmode='data')
fig.show()

In [115]:
candidatos=[2116, 44596, 36359 ,4265, 423]


In [116]:
dfss.loc[dfss.indice.isin(candidatos),['indice','outlier']]

,indice,outlier
2347,2116,1
4689,44596,1
5114,36359,1
6224,4265,1
6683,423,1


## 2.2 ENVOLVENTE CONVEXA

In [117]:
def profundidad_convexa(datos,kmin):
    k=1
    copia=datos.values
    n,m=copia.shape
    depths=np.zeros(n)
    indices=np.array(range(len(depths)))
    while len(copia)>m: #En función ConvexHull, no se puede calcular envolvente sin al menos m+1 puntos (m es la dimensión)
        hull=ConvexHull(copia)
        depths[indices[hull.vertices]]=k
        if k==kmin:
            frontera=indices[hull.vertices]
        copia=np.delete(copia,hull.vertices,axis=0)
        indices=np.delete(indices,hull.vertices)
        k+=1
    depths[indices]=k #Si quedan puntos por asignar, estos tienen la máxima profundidad
    return depths,frontera

In [118]:
depths,frontera=profundidad_convexa(datos,5)
dfss["profundidad"]=depths

In [119]:
fig = px.scatter_3d(dfss, x='SE1', y='SE2', z='SE3',color='profundidad',opacity=0.7,hover_data=['clase','indice'])
scene=dict(aspectratio=dict(x=1, y=1,z=1))
fig.update_traces(marker_size = 2)
fig.update_scenes(aspectmode='data')
fig.show()

In [120]:
dfss.loc[:,'outlier']=1
dfss.loc[dfss.profundidad==1,'outlier']=-1
fig = px.scatter_3d(dfss, x='SE1', y='SE2', z='SE3',color='outlier',opacity=0.7,hover_data=['clase','indice'])
scene=dict(aspectratio=dict(x=1, y=1,z=1))
fig.update_traces(marker_size = 2)
fig.update_scenes(aspectmode='data')
fig.show()

In [121]:
dfss.loc[dfss.indice.isin(candidatos),['indice','outlier']]

,indice,outlier
2347,2116,-1
4689,44596,-1
5114,36359,-1
6224,4265,-1
6683,423,-1


## 2.3 KERNEL DENSITY ESTIMATION

In [122]:
estimator=KDE(contamination=0.05)
X=datos.values
estimator.fit(X)
dfss['outlier']=estimator.predict(X)
dfss['outlier']=1-2*dfss['outlier'].values #Para que el número -1 sea los outliers
fig = px.scatter_3d(dfss, x='SE1', y='SE2', z='SE3',color='outlier',opacity=0.7,hover_data=['clase','indice'])
scene=dict(aspectratio=dict(x=1, y=1,z=1))
fig.update_traces(marker_size = 2)
fig.update_scenes(aspectmode='data')
fig.show()

In [123]:
dfss.loc[dfss.indice.isin(candidatos),['indice','outlier']]

,indice,outlier
2347,2116,-1
4689,44596,-1
5114,36359,-1
6224,4265,-1
6683,423,-1


## 2.4 ALGORITMO ECOD

In [124]:
estimator=ECOD(contamination=0.02)
X=datos.values
estimator.fit(X)
dfss['outlier']=estimator.predict(X)
dfss['outlier']=1-2*dfss['outlier'].values #Para que el número -1 sea los outliers
fig = px.scatter_3d(dfss, x='SE1', y='SE2', z='SE3',color='outlier',opacity=0.7,hover_data=['clase','indice'])
scene=dict(aspectratio=dict(x=1, y=1,z=1))
fig.update_traces(marker_size = 2)
fig.update_scenes(aspectmode='data')
fig.show()

In [125]:
dfss.loc[dfss.indice.isin(candidatos),['indice','outlier']]

,indice,outlier
2347,2116,-1
4689,44596,-1
5114,36359,-1
6224,4265,-1
6683,423,-1


## 2.5 SVM DE UNA CLASE

In [126]:
estimator=OneClassSVM(gamma='scale',nu=0.01)
X=datos.values
estimator.fit(X)
dfss['outlier']=estimator.predict(X)
fig = px.scatter_3d(dfss, x='SE1', y='SE2', z='SE3',color='outlier',opacity=0.7,hover_data=['clase','indice'])
scene=dict(aspectratio=dict(x=1, y=1,z=1))
fig.update_traces(marker_size = 2)
fig.update_scenes(aspectmode='data')
fig.show()

In [127]:
dfss.loc[dfss.indice.isin(candidatos),['indice','outlier']]

,indice,outlier
2347,2116,-1
4689,44596,-1
5114,36359,-1
6224,4265,-1
6683,423,-1


# 3. DISCUCIÓN DATOS ANOMALOS

In [161]:
candidatos=[2116, 44596 ,4265,38461]


In [163]:
letras=['A','B','C','D']

def add_mark(fig,dfc,indx,color,hlabel):
    xx,yy,zz=dfc.loc[dfc.indice==indx,['SE1','SE2','SE3']].values[0,:]
    fig.add_trace(go.Scatter3d(x=[xx], y=[yy],z=[zz],marker=dict(size=4,color=color),hovertext=hlabel,showlegend=False))
    fig.add_trace(go.Scatter3d(x=[xx], y=[yy],z=[zz],mode="text",text=hlabel,showlegend=False))
fig = px.scatter_3d(dfss, x='SE1', y='SE2', z='SE3',color='clase',opacity=0.7,hover_data=['clase','indice'])
scene=dict(aspectratio=dict(x=1, y=1,z=1))
fig.update_traces(marker_size = 2)
fig.update_scenes(aspectmode='data')
for k in range(4):
    add_mark(fig,dfss,candidatos[k],"green","Punto "+letras[k])
fig.show()

## 3.1 PRIMER ANOMALO

In [147]:
dfss.loc[dfss.indice==candidatos[0],:]

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q,outlier,profundidad
2347,defensa,Ejecutivo,Centralizada,Prestación de servicios,Mínima cuantía,No,Si,No,Si,No,No,No Válido,No,Si,2,3,1,2,0.201081,1.381923,-0.101329,-0.019134,-0.448056,-0.017657,-0.1349,-0.041319,0.022469,-0.068705,-0.191619,0.350309,0.331975,-0.204866,0.078309,-0.486484,0.126389,-0.522498,-0.035933,0.22323,0.076906,0.525059,-0.201145,-0.202871,-0.339173,0.256922,-0.043857,-0.012546,0.083169,0.503831,-0.235153,0.131832,-0.343534,0.203429,-0.279923,1.551939,2.318834,-0.785418,2,2116,2,3,1,2,-1,1.0


El contrato fue identificado como un caso atípico porque presenta características poco comunes frente a la mayoría de contratos del dataset. Aunque pertenece a la modalidad de mínima cuantía, tiene una duración muy larga de 358 días y además fue prorrogado, algo que normalmente no ocurre en este tipo de contratos. También incluye pago adelantado y obligaciones ambientales, lo que hace que su comportamiento sea diferente al de los demás registros analizados. Por esta razón, el modelo lo detectó como un contrato alejado del patrón general de los datos.

## 3.2 SEGUNDO ANOMALO

In [148]:
dfss.loc[dfss.indice==candidatos[1],:]

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q,outlier,profundidad
4689,Salud y Protección Social,Ejecutivo,Centralizada,Decreto 092 de 2017,Contratación régimen especial,No,No,No,No,No,No,No Válido,No,No,0,1,2,1,2.207266,-0.594508,0.014677,0.016784,0.08915,0.355825,0.003395,-0.791883,-0.092761,0.121742,0.125767,-0.206243,-0.368539,0.087649,0.00959,0.550968,0.087245,0.071338,0.00545,-0.072339,0.011376,-0.038077,-0.210856,-0.085247,-0.04059,-0.044634,0.122244,0.042276,-0.120796,0.070921,-0.106539,-0.011747,-0.168933,0.154766,0.097665,-0.857302,0.447303,3.005442,1,44596,0,1,2,1,-1,1.0


Este contrato fue identificado como un caso atípico porque presenta características diferentes a las que normalmente aparecen en el conjunto de datos. Aunque tiene un valor relativamente moderado y una duración corta de 87 días, pertenece al tipo “Decreto 092 de 2017” bajo contratación de régimen especial, una modalidad menos frecuente dentro de los registros analizados. Además, algunas de sus variables transformadas muestran valores bastante alejados del promedio, especialmente en las componentes MCA y en SE3, lo que indica que el contrato se comporta de manera distinta frente a los demás. Por esta razón, el modelo lo detectó como un registro fuera del patrón general de contratación.

## 3.3 TERCER ANOMALO

In [149]:
dfss.loc[dfss.indice==candidatos[2],:]

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q,outlier,profundidad
5114,Servicio Público,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,2,3,3,2,-0.442577,-0.190577,0.004161,0.002818,0.02669,0.185955,-0.043467,-0.162269,-0.052908,0.073032,0.046968,0.133886,-0.0402,-0.095047,0.021912,0.121379,0.022787,-0.088817,0.051312,0.06866,0.055401,-0.151738,0.00432,-0.078546,0.073219,0.072032,0.195943,-0.081875,-0.01478,-0.219008,-0.012906,0.208623,-0.122016,0.21252,0.030249,-0.857972,0.464175,3.023987,1,36359,2,3,3,2,-1,1.0


Este contrato fue identificado como un caso atípico porque, aunque en apariencia tiene características normales dentro del conjunto de datos, presenta un comportamiento diferente en las variables transformadas utilizadas por el modelo. El contrato pertenece al sector de servicio público y fue realizado por contratación directa bajo prestación de servicios, con una duración relativamente alta de 294 días. Además, algunas de sus componentes MCA y especialmente la variable SE3 muestran valores alejados del comportamiento promedio, lo que indica que el registro se ubica en una zona poco común dentro de los datos analizados. Por esta razón, el modelo lo clasificó como un contrato fuera del patrón general.

## 3.4 CUARTO ANOMALO

In [150]:
dfss.loc[dfss.indice==candidatos[3],:]

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q,outlier,profundidad
6224,Servicio Público,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,No Válido,No,No,2,2,1,2,-0.397731,-0.197147,0.005842,0.006077,0.016604,0.148651,-0.036331,-0.145595,-0.000919,0.052624,0.076648,0.179538,-0.013848,-0.142886,0.058698,0.08893,0.028534,-0.112758,0.075234,0.082162,0.080118,-0.156285,0.005067,-0.117989,0.065349,0.109027,0.241389,-0.095134,-0.028666,-0.236057,-0.037698,0.281472,-0.170756,0.289937,0.022665,1.780232,-2.057557,0.805376,3,4265,2,2,1,2,-1,1.0


Este contrato fue identificado como un caso atípico porque presenta un comportamiento diferente al de la mayoría de contratos del dataset. Aunque corresponde a una contratación directa por prestación de servicios, tiene características que lo hacen poco común, como un tiempo de espera de 24 días antes del inicio y un estado BPIN “No válido”. Además, las variables transformadas del modelo, especialmente en las componentes SE1 y SE2, muestran valores bastante alejados del promedio, indicando que el contrato se encuentra en una zona poco frecuente dentro de los datos analizados. Por esta razón, el modelo lo detectó como un registro fuera del patrón general.

## 3.5 QUINTO ANOMALO

In [151]:
dfss.loc[dfss.indice==candidatos[4],:]

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q,outlier,profundidad
7520,Planeación,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,No Válido,No,No,2,0,3,0,-0.236356,-0.232543,0.521578,0.011766,0.01768,-0.14601,-0.040653,0.116006,0.033959,0.089594,0.164528,0.536736,-0.06405,-0.068518,-0.105303,0.237339,0.021603,-0.133762,-0.081177,0.367218,0.334643,-0.309987,0.356876,0.098482,0.516013,0.136842,-3.309283,-0.183172,-1.2584,-0.260693,0.931835,0.583255,-0.748228,0.403769,-0.53831,-0.619335,-0.780406,-1.317176,0,38461,2,0,3,0,-1,1.0


Este contrato fue identificado como un caso atípico porque presenta algunas características que no son tan comunes dentro del conjunto de datos. Aunque pertenece a una contratación directa por prestación de servicios y tiene un valor relativamente bajo, el contrato fue prorrogado y cuenta con una duración de 228 días, lo que puede diferenciarlo de otros contratos similares. Además, las variables transformadas del modelo muestran valores alejados del comportamiento promedio, especialmente en las componentes SE2 y SE3, indicando que el registro se encuentra en una zona menos frecuente dentro de los datos. Por esta razón, el modelo lo clasificó como un contrato fuera del patrón general.

## 3.6 SEXTO ANOMALO

In [ ]:
dfss.loc[dfss.indice==candidatos[5],:]

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,outlier,profundidad
6683,Servicio Público,Ejecutivo,Descentralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,Si,1,90,2875,10919700,-0.346254,-0.168691,0.000497,-0.00034,0.017922,0.252364,-0.089119,-0.164326,-0.068432,0.048707,-0.014563,0.089199,0.109629,-0.08568,0.106642,-0.12025,0.073898,-0.124085,0.021832,0.147175,0.127425,-0.04532,-0.033046,-0.18353,0.042758,0.141752,0.17929,-0.0254,0.023365,-0.257105,-0.057093,0.249656,-0.108987,0.182913,-0.026352,1.547129,2.318334,-0.788547,2,423,-1,1.0


Este contrato fue identificado como un caso atípico porque presenta un comportamiento diferente frente a la mayoría de registros analizados. Aunque corresponde a una contratación directa por prestación de servicios, pertenece a una entidad descentralizada y además fue prorrogado, aun teniendo una duración relativamente corta de 90 días y un valor bajo del contrato. También se observa que las variables transformadas del modelo, especialmente en las componentes SE1 y SE2, toman valores alejados del promedio, indicando que el contrato se ubica en una zona poco común dentro de los datos. Por esta razón, el modelo lo detectó como un registro fuera del patrón general.

# 4. MINERÍA DE REGLAS DE ASOCIACIÓN

In [26]:
# Variables binarias (sí/no, verdadero/falso)
binarias = ['centralizada','grupo','pime','pago_adelantado','liquidacion','obligacion_ambiente','obligacion_postconsumo','estado_bpin','prorrogado','postconflicto']

# Variables categóricas (sin un orden específico)
cuals = ['sector','rama','tipo_contrato','modalidad_contratacion']

# Variables cuantitativas (numéricas que representan cantidades medibles)
cuants = ['dias_espera_inicio','duracion_contrato','dias_desde_primer_contrato','valor_contratoo']

In [27]:

transformer = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile',subsample=None)
dfs.loc[:,cuants]=transformer.fit_transform(dfs.loc[:,cuants]) #Las cuantitativas las transformamos a cuartiles
for cuant in cuants:
     dfs.loc[:,cuant+'_q']=dfs.loc[:,cuant].apply(int).astype('str') #Para evitar un warning de pandas se crean nuevas columnas con cuantiles y dtype object

c:\Users\alcas\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning:

The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.

c:\Users\alcas\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning:

Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.



## 4.1 MINERÍA DE LA CLASE 0

### 4.1.0 SUBCONJUNTO Y PREPRACIÓN

In [177]:
df0=dfs.copy()[dfs.clase==0]
df0.head()

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q
0,Transporte,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,Si,2,2,1,1,-0.230632,-0.122036,-0.001522,-0.002303,0.134220,0.176406,-0.056739,-0.030522,-0.149470,0.164997,-0.101998,-0.269728,0.280453,0.097403,0.188180,-0.512083,0.142109,0.154056,-0.289179,0.131029,0.087097,0.158542,0.065064,0.098843,0.241269,-0.301444,-0.155364,-0.045086,0.110323,-0.290308,-0.021428,-0.183431,-0.165765,0.050169,0.516933,-0.699410,-0.640868,-1.197668,0,0,2,2,1,1
3,Educación Nacional,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,1,0,2,1,-0.338042,-0.278063,0.010924,0.013671,-0.009648,-0.363740,-0.090634,0.265894,0.020466,-0.003160,-0.173324,0.609988,-0.310310,-0.457564,-0.354598,-0.334695,-0.015035,0.882740,0.052598,-0.168797,-0.091880,0.185196,-0.353456,-0.112787,0.114576,-0.231244,0.083896,0.326041,-0.051403,0.082964,-0.121959,-0.043341,-0.058637,0.063921,0.094111,-0.332191,-0.062011,-0.105104,0,3,1,0,2,1
4,Inclusión Social y Reconciliación,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,2,3,1,2,-0.487375,-0.318814,0.011714,0.008560,0.014443,0.285101,-0.016738,-0.257863,-0.212083,0.107955,0.124422,-0.003640,-0.141943,0.141153,-0.214277,0.193188,0.021567,-0.002867,-0.011424,-0.052882,0.066520,-0.447140,0.143064,0.269374,0.558160,-0.470381,0.162990,-0.043987,-0.366043,1.742416,-0.265758,-0.090741,0.681484,0.093808,-0.622647,-0.693104,0.502032,0.551954,0,4,2,3,1,2
5,Salud y Protección Social,Corporación Autónoma,Centralizada,Prestación de servicios,Contratación régimen especial,No,No,No,No,No,No,No Válido,No,Si,0,0,0,0,1.382267,-0.516775,0.015610,0.014045,0.045300,-0.208494,-0.046099,0.097162,-0.005489,-0.062138,-0.039359,0.025348,0.298700,-0.021544,0.093522,-0.499560,0.159782,-0.089353,-0.091828,0.073691,0.004596,0.092507,0.100983,-0.101308,0.070730,0.035983,-0.050008,-0.024554,0.125461,-0.024174,0.014963,0.043829,0.054716,-0.002287,-0.070030,-0.541255,0.524765,0.174724,0,5,0,0,0,0
6,deportes,Ejecutivo,Descentralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,No Válido,No,No,0,3,1,2,-0.504574,-0.397166,0.010994,0.009706,0.000229,-0.068255,-0.089499,-0.212740,-0.239479,0.153282,0.305977,0.096343,-0.213860,0.181756,-0.390827,0.653696,0.209369,-0.019500,-0.699967,-0.678877,-1.690219,1.256887,0.835796,-0.485599,-0.379894,0.836112,-0.060325,-0.125866,-0.101270,0.063530,-0.101731,0.148275,0.291683,-0.147344,-0.028407,-0.634507,-0.742000,-1.152330,0,6,0,3,1,2


In [178]:
df_onehot=pd.concat([pd.get_dummies(df0.loc[:,cuals+[cuant+'_q' for cuant in cuants]],drop_first=False),pd.get_dummies(df0.loc[:,binarias],drop_first=True)],axis=1)
df_onehot.head()

,sector_Ambiente y Desarrollo Sostenible,sector_Ciencia Tecnología,sector_Cultura,sector_Educación Nacional,sector_Hacienda y Crédito Público,sector_Inclusión Social y Reconciliación,sector_Industria,sector_Información Estadística,sector_Inteligencia Estratégica y Contrainteligencia,sector_Ley de Justicia,sector_Minas y Energía,sector_No aplica/No pertenece,sector_Planeación,sector_Presidencia de la República,sector_Relaciones Exteriores,sector_Salud y Protección Social,sector_Servicio Público,sector_Tecnologías de la Información y las Comunicaciones,sector_Trabajo,sector_Transporte,"sector_Vivienda, Ciudad y Territorio",sector_agricultura,sector_defensa,sector_deportes,sector_interior,rama_Corporación Autónoma,rama_Ejecutivo,rama_Judicial,rama_Legislativo,tipo_contrato_Acuerdo Marco de Precios,tipo_contrato_Arrendamiento de inmuebles,tipo_contrato_Arrendamiento de muebles,tipo_contrato_Comisión,tipo_contrato_Comodato,tipo_contrato_Compraventa,tipo_contrato_Concesión,tipo_contrato_Consultoría,tipo_contrato_Decreto 092 de 2017,tipo_contrato_Interventoría,tipo_contrato_Negocio fiduciario,tipo_contrato_No Especificado,tipo_contrato_Obra,tipo_contrato_Operaciones de Crédito Público,tipo_contrato_Otro,tipo_contrato_Prestación de servicios,tipo_contrato_Seguros,tipo_contrato_Servicios financieros,tipo_contrato_Suministros,tipo_contrato_Venta muebles,modalidad_contratacion_Concurso de méritos abierto,modalidad_contratacion_Contratación Directa (con ofertas),modalidad_contratacion_Contratación directa,modalidad_contratacion_Contratación régimen especial,modalidad_contratacion_Contratación régimen especial (con ofertas),modalidad_contratacion_Enajenación de bienes con subasta,modalidad_contratacion_Licitación Pública Acuerdo Marco de Precios,modalidad_contratacion_Licitación pública,modalidad_contratacion_Licitación pública Obra Publica,modalidad_contratacion_Mínima cuantía,modalidad_contratacion_Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes,modalidad_contratacion_Selección Abreviada de Menor Cuantía,modalidad_contratacion_Selección abreviada subasta inversa,dias_espera_inicio_q_0,dias_espera_inicio_q_1,dias_espera_inicio_q_2,duracion_contrato_q_0,duracion_contrato_q_1,duracion_contrato_q_2,duracion_contrato_q_3,dias_desde_primer_contrato_q_0,dias_desde_primer_contrato_q_1,dias_desde_primer_contrato_q_2,dias_desde_primer_contrato_q_3,valor_contratoo_q_0,valor_contratoo_q_1,valor_contratoo_q_2,valor_contratoo_q_3,centralizada_Descentralizada,grupo_Si,pime_Si,pago_adelantado_Si,liquidacion_Si,obligacion_ambiente_Si,obligacion_postconsumo_Si,estado_bpin_Válido,prorrogado_Si,postconflicto_Si
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True,True,False
3,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False
4,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,Fal

In [179]:
df_onehot.sum(axis=0)/df_onehot.shape[0]

sector_Ambiente y Desarrollo Sostenible                                               0.040232
sector_Ciencia Tecnología                                                             0.002560
sector_Cultura                                                                        0.027237
sector_Educación Nacional                                                             0.053720
sector_Hacienda y Crédito Público                                                     0.009910
sector_Inclusión Social y Reconciliación                                              0.024678
sector_Industria                                                                      0.011486
sector_Información Estadística                                                        0.030059
sector_Inteligencia Estratégica y Contrainteligencia                                  0.000033
sector_Ley de Justicia                                                                0.015063
sector_Minas y Energía                            

In [180]:
cumplen=(df_onehot.sum(axis=0)/df_onehot.shape[0]>0.1) & (df_onehot.sum(axis=0)/df_onehot.shape[0]<0.9)
df_onehot=df_onehot.loc[:,cumplen]
df_onehot.head()

,sector_No aplica/No pertenece,sector_Salud y Protección Social,sector_Servicio Público,rama_Corporación Autónoma,rama_Ejecutivo,tipo_contrato_Prestación de servicios,modalidad_contratacion_Contratación directa,modalidad_contratacion_Contratación régimen especial,dias_espera_inicio_q_0,dias_espera_inicio_q_1,dias_espera_inicio_q_2,duracion_contrato_q_0,duracion_contrato_q_1,duracion_contrato_q_2,duracion_contrato_q_3,dias_desde_primer_contrato_q_0,dias_desde_primer_contrato_q_1,dias_desde_primer_contrato_q_2,dias_desde_primer_contrato_q_3,valor_contratoo_q_0,valor_contratoo_q_1,valor_contratoo_q_2,valor_contratoo_q_3,centralizada_Descentralizada,pime_Si,liquidacion_Si,estado_bpin_Válido,prorrogado_Si
0,False,False,False,False,True,True,True,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,True,True
3,False,False,False,False,True,True,True,False,False,True,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False
4,False,False,False,False,True,True,True,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,True,False
5,False,True,False,True,False,True,False,True,True,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True
6,False,False,False,False,True,True,True,False,True,False,False,False,False,False,True,False,True,False,False,False,False,True,False,True,False,False,False,False


### 4.1.1 MINERÍA DE ITEMSETS FRECUENTES

In [181]:
freqitems = apriori(df_onehot, min_support=0.08, use_colnames=True)
freqitems.sort_values('support',ascending=True)

,support,itemsets
160,0.080432,"frozenset({centralizada_Descentralizada, estad..."
41,0.080465,"frozenset({sector_Servicio Público, duracion_c..."
361,0.080530,"frozenset({dias_espera_inicio_q_1, tipo_contra..."
166,0.080662,frozenset({modalidad_contratacion_Contratación...
87,0.080760,frozenset({tipo_contrato_Prestación de servici...
435,0.081023,frozenset({tipo_contrato_Prestación de servici...
118,0.081187,"frozenset({dias_espera_inicio_q_1, duracion_co..."
152,0.081252,"frozenset({valor_contratoo_q_1, dias_desde_pri..."
165,0.081285,frozenset({tipo_contrato_Prestación de servici...
126,0.081318,"frozenset({dias_espera_inicio_q_1, valor_contr..."


In [182]:
freqitems =fpgrowth(df_onehot, min_support=0.08, use_colnames=True)
freqitems.sort_values('support',ascending=True).head(300)

,support,itemsets
340,0.080432,"frozenset({centralizada_Descentralizada, estad..."
435,0.080465,"frozenset({sector_Servicio Público, duracion_c..."
118,0.080530,"frozenset({dias_espera_inicio_q_1, tipo_contra..."
355,0.080662,frozenset({modalidad_contratacion_Contratación...
456,0.080760,frozenset({tipo_contrato_Prestación de servici...
171,0.081023,frozenset({tipo_contrato_Prestación de servici...
146,0.081187,"frozenset({dias_espera_inicio_q_1, duracion_co..."
180,0.081252,"frozenset({valor_contratoo_q_1, dias_desde_pri..."
354,0.081285,frozenset({tipo_contrato_Prestación de servici...
271,0.081318,"frozenset({dias_espera_inicio_q_1, valor_contr..."


In [183]:
freqitems.shape

(457, 2)

### 4.1.2 MINERÍA DE REGLAS DE ASOCIACIÓN

In [184]:
rules = association_rules(freqitems, metric="lift", min_threshold=1.5)

rules.loc[:, "pearson"] = rules["leverage"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"] *
    (1 - rules["antecedent support"]) * (1 - rules["consequent support"])
)

rules.loc[:, "cosine"] = rules["support"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"]
)

rules.loc[:, "jaccard"] = rules["support"] / (
    rules["antecedent support"] + rules["consequent support"] - rules["support"]
)

rules.loc[:, "kuczynski"] = (rules["support"] / 2) * (
    1 / rules["antecedent support"] + 1 / rules["consequent support"]
)

rules.loc[:, "certainty"] = (
    rules["confidence"] - rules["consequent support"]
) / (1 - rules["consequent support"])

rules.head(100)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,pearson,cosine,kuczynski
0,frozenset({valor_contratoo_q_0}),frozenset({duracion_contrato_q_0}),0.250878,0.246152,0.148525,0.592021,2.405100,1.0,0.086771,1.847761,0.779869,0.426177,0.458804,0.597704,0.464646,0.597677,0.597704
1,frozenset({duracion_contrato_q_0}),frozenset({valor_contratoo_q_0}),0.246152,0.250878,0.148525,0.603386,2.405100,1.0,0.086771,1.888795,0.774980,0.426177,0.470562,0.597704,0.464646,0.597677,0.597704
2,frozenset({duracion_contrato_q_0}),frozenset({dias_desde_primer_contrato_q_3}),0.246152,0.254619,0.149181,0.606053,2.380234,1.0,0.086506,1.892083,0.769218,0.424305,0.471482,0.595976,0.460966,0.595891,0.595976
3,frozenset({dias_desde_primer_contrato_q_3}),frozenset({duracion_contrato_q_0}),0.254619,0.246152,0.149181,0.585900,2.380234,1.0,0.086506,1.820449,0.777955,0.424305,0.450685,0.595976,0.460966,0.595891,0.595976
4,"frozenset({valor_contratoo_q_0, modalidad_cont...",frozenset({duracion_contrato_q_0}),0.183704,0.246152,0.102484,0.557878,2.266393,1.0,0.057265,1.705066,0.684519,0.313051,0.413513,0.487111,0.343292,0.481943,0.487111
5,"frozenset({duracion_contrato_q_0, modalidad_co...",frozenset({valor_contratoo_q_0}),0.147639,0.250878,0.102484,0.694154,2.766902,1.0,0.065445,2.449346,0.749195,0.346192,0.591728,0.551328,0.425555,0.532507,0.551328
6,frozenset({valor_contratoo_q_0}),"frozenset({duracion_contrato_q_0, modalidad_co...",0.250878,0.147639,0.102484,0.408502,2.766902,1.0,0.065445,1.441022,0.852444,0.346192,0.306048,0.551328,0.425555,0.532507,0.551328
7,frozenset({duracion_contrato_q_0}),"frozenset({valor_contratoo_q_0, modalidad_cont...",0.246152,0.183704,0.102484,0.416344,2.266393,1.0,0.057265,1.398593,0.741224,0.313051,0.284996,0.487111,0.343292,0.481943,0.487111
8,"frozenset({duracion_contrato_q_0, modalidad_co...",frozenset({dias_desde_primer_contrato_q_3}),0.147639,0.254619,0.096216,0.651700,2.559514,1.0,0.058625,2.140057,0.714839,0.314390,0.532723,0.514792,0.379345,0.496253,0.514792
9,frozenset({modalidad_contratacion_Contratación...,frozenset({duracion_contrato_q_0}),0.175237,0.246152,0.096216,0.549064,2.230585,1.0,0.053081,1.671739,0.668904,0.295893,0.401820,0.469972,0.324131,0.463270,0.469972


In [190]:
rules_sub=rules[(rules.confidence > 0.5)&(rules.zhangs_metric > 0.5)].sort_values(by='lift',ascending=False)
rules_sub.head(100)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,pearson,cosine,kuczynski
297,frozenset({sector_Salud y Protección Social}),frozenset({modalidad_contratacion_Contratación...,0.150658,0.150527,0.110688,0.734698,4.880851,1.0,0.088010,3.201915,0.936157,0.581051,0.687687,0.735019,0.688040,0.735019,0.735019
296,frozenset({modalidad_contratacion_Contratación...,frozenset({sector_Salud y Protección Social}),0.150527,0.150658,0.110688,0.735339,4.880851,1.0,0.088010,3.209170,0.936013,0.581051,0.688393,0.735019,0.688040,0.735019,0.735019
89,"frozenset({valor_contratoo_q_0, dias_desde_pri...",frozenset({tipo_contrato_Prestación de servici...,0.115676,0.186296,0.081023,0.700426,3.759744,1.0,0.059473,2.716199,0.830040,0.366701,0.631838,0.567669,0.477588,0.551928,0.567669
289,"frozenset({valor_contratoo_q_3, dias_desde_pri...",frozenset({duracion_contrato_q_3}),0.095954,0.249696,0.082565,0.860465,3.446045,1.0,0.058606,5.377175,0.785150,0.313833,0.814029,0.595563,0.459714,0.533406,0.595563
350,"frozenset({valor_contratoo_q_3, modalidad_cont...",frozenset({tipo_contrato_Prestación de servici...,0.155285,0.179897,0.091950,0.592139,3.291543,1.0,0.064015,2.010740,0.824173,0.378036,0.502671,0.551633,0.460167,0.550144,0.551633
341,frozenset({tipo_contrato_Prestación de servici...,"frozenset({valor_contratoo_q_3, modalidad_cont...",0.179897,0.155285,0.091950,0.511127,3.291543,1.0,0.064015,1.727883,0.848907,0.378036,0.421257,0.551633,0.460167,0.550144,0.551633
340,frozenset({tipo_contrato_Prestación de servici...,frozenset({modalidad_contratacion_Contratación...,0.145736,0.192137,0.091950,0.630939,3.283792,1.0,0.063949,2.188968,0.814120,0.373899,0.543164,0.554752,0.460024,0.549496,0.554752
41,"frozenset({duracion_contrato_q_0, modalidad_co...","frozenset({rama_Ejecutivo, tipo_contrato_Prest...",0.147639,0.172415,0.083320,0.564348,3.273195,1.0,0.057865,1.899646,0.814782,0.351955,0.473586,0.523799,0.431824,0.522227,0.523799
344,"frozenset({rama_Ejecutivo, valor_contratoo_q_3...",frozenset({tipo_contrato_Prestación de servici...,0.132872,0.211827,0.091950,0.692023,3.266926,1.0,0.063804,2.559192,0.800230,0.363802,0.609252,0.563052,0.460036,0.548083,0.563052
325,frozenset({tipo_contrato_Prestación de servici...,"frozenset({valor_contratoo_q_3, modalidad_cont...",0.211827,0.155285,0.106258,0.501627,3.230361,1.0,0.073364,1.694944,0.875997,0.407347,0.410010,0.592952,0.495752,0.585877,0.592952


In [186]:
vals=[len(list(rules_sub.iloc[k,:].consequents)) == 1 for k in range(rules_sub.shape[0])]

In [187]:
np.array(vals).sum()

np.int64(80)

In [188]:
rules_sub.loc[vals,:].consequents.drop_duplicates()

297    frozenset({modalidad_contratacion_Contratación...
296        frozenset({sector_Salud y Protección Social})
289                   frozenset({duracion_contrato_q_3})
83                    frozenset({duracion_contrato_q_0})
29                      frozenset({valor_contratoo_q_0})
290                     frozenset({valor_contratoo_q_3})
52           frozenset({dias_desde_primer_contrato_q_3})
264          frozenset({dias_desde_primer_contrato_q_0})
299                  frozenset({dias_espera_inicio_q_0})
Name: consequents, dtype: object

In [189]:
vals=[list(rules_sub.iloc[k,:].consequents) == ['duracion_contrato_q_0'] for k in range(rules_sub.shape[0])]
print(rules_sub.loc[vals,:].iloc[0,:].antecedents)
rules_sub.loc[vals,:].iloc[0,:]

frozenset({'tipo_contrato_Prestación de servicios', 'valor_contratoo_q_0', 'dias_desde_primer_contrato_q_3'})


antecedents           frozenset({tipo_contrato_Prestación de servici...
consequents                          frozenset({duracion_contrato_q_0})
antecedent support                                             0.103961
consequent support                                             0.246152
support                                                        0.081023
confidence                                                     0.779356
lift                                                           3.166153
representativity                                                    1.0
leverage                                                       0.055432
conviction                                                      3.41658
zhangs_metric                                                  0.763537
jaccard                                                        0.301098
certainty                                                       0.70731
kulczynski                                                     0

In [191]:
cols = [
    'antecedents', 'consequents',
    'antecedent support', 'consequent support',
    'support', 'confidence', 'lift', 'zhangs_metric'
]

seleccion_clase0 = rules.loc[[0, 17, 83], cols].copy()

def limpiar_itemset(x):
    return ' + '.join(list(x))

seleccion_clase0['antecedents'] = seleccion_clase0['antecedents'].apply(limpiar_itemset)
seleccion_clase0['consequents'] = seleccion_clase0['consequents'].apply(limpiar_itemset)

seleccion_clase0

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,zhangs_metric
0,valor_contratoo_q_0,duracion_contrato_q_0,0.250878,0.246152,0.148525,0.592021,2.405100,0.779869
17,duracion_contrato_q_0 + modalidad_contratacion...,tipo_contrato_Prestación de servicios + valor_...,0.147639,0.217176,0.100646,0.681707,3.138963,0.799454
83,tipo_contrato_Prestación de servicios + valor_...,duracion_contrato_q_0,0.103961,0.246152,0.081023,0.779356,3.166153,0.763537


## 4.2 MINERÍA DE LA CLASE 1

In [192]:
df0=dfs.copy()[dfs.clase==1]
df_onehot=pd.concat([pd.get_dummies(df0.loc[:,cuals+[cuant+'_q' for cuant in cuants]],drop_first=False),pd.get_dummies(df0.loc[:,binarias],drop_first=True)],axis=1)
cumplen=(df_onehot.sum(axis=0)/df_onehot.shape[0]>0.1) & (df_onehot.sum(axis=0)/df_onehot.shape[0]<0.9)
df_onehot=df_onehot.loc[:,cumplen]

In [193]:
freqitems =fpgrowth(df_onehot, min_support=0.08, use_colnames=True)

In [194]:
rules = association_rules(freqitems, metric="lift", min_threshold=1.5)

rules.loc[:, "pearson"] = rules["leverage"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"] *
    (1 - rules["antecedent support"]) * (1 - rules["consequent support"])
)

rules.loc[:, "cosine"] = rules["support"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"]
)

rules.loc[:, "jaccard"] = rules["support"] / (
    rules["antecedent support"] + rules["consequent support"] - rules["support"]
)

rules.loc[:, "kuczynski"] = (rules["support"] / 2) * (
    1 / rules["antecedent support"] + 1 / rules["consequent support"]
)

rules.loc[:, "certainty"] = (
    rules["confidence"] - rules["consequent support"]
) / (1 - rules["consequent support"])

rules.head(100)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,pearson,cosine,kuczynski
0,frozenset({valor_contratoo_q_2}),frozenset({duracion_contrato_q_2}),0.247317,0.258567,0.104361,0.421973,1.631969,1.0,0.040413,1.282697,0.514484,0.259914,0.220393,0.412794,0.213928,0.412692,0.412794
1,frozenset({duracion_contrato_q_2}),frozenset({valor_contratoo_q_2}),0.258567,0.247317,0.104361,0.403614,1.631969,1.0,0.040413,1.262074,0.522291,0.259914,0.207653,0.412794,0.213928,0.412692,0.412794
2,"frozenset({valor_contratoo_q_2, tipo_contrato_...",frozenset({duracion_contrato_q_2}),0.224645,0.258567,0.100554,0.447612,1.731125,1.0,0.042468,1.342232,0.544707,0.262777,0.254972,0.418250,0.232402,0.417218,0.418250
3,frozenset({tipo_contrato_Prestación de servici...,frozenset({valor_contratoo_q_2}),0.233299,0.247317,0.100554,0.431009,1.742736,1.0,0.042855,1.322837,0.555874,0.264572,0.244049,0.418793,0.234854,0.418615,0.418793
4,frozenset({valor_contratoo_q_2}),frozenset({tipo_contrato_Prestación de servici...,0.247317,0.233299,0.100554,0.406578,1.742736,1.0,0.042855,1.292000,0.566227,0.264572,0.226006,0.418793,0.234854,0.418615,0.418793
5,frozenset({duracion_contrato_q_2}),"frozenset({valor_contratoo_q_2, tipo_contrato_...",0.258567,0.224645,0.100554,0.388889,1.731125,1.0,0.042468,1.268762,0.569628,0.262777,0.211830,0.418250,0.232402,0.417218,0.418250
6,"frozenset({valor_contratoo_q_2, modalidad_cont...",frozenset({duracion_contrato_q_2}),0.200242,0.258567,0.094669,0.472774,1.828441,1.0,0.042893,1.406292,0.566529,0.259981,0.288910,0.419453,0.244800,0.416050,0.419453
7,frozenset({modalidad_contratacion_Contratación...,frozenset({valor_contratoo_q_2}),0.213742,0.247317,0.094669,0.442915,1.790877,1.0,0.041807,1.351109,0.561666,0.258385,0.259867,0.412850,0.236370,0.411754,0.412850
8,frozenset({valor_contratoo_q_2}),frozenset({modalidad_contratacion_Contratación...,0.247317,0.213742,0.094669,0.382785,1.790877,1.0,0.041807,1.273881,0.586720,0.258385,0.214997,0.412850,0.236370,0.411754,0.412850
9,frozenset({duracion_contrato_q_2}),"frozenset({valor_contratoo_q_2, modalidad_cont...",0.258567,0.200242,0.094669,0.366131,1.828441,1.0,0.042893,1.261709,0.611095,0.259981,0.207424,0.419453,0.244800,0.416050,0.419453


In [195]:
rules_sub=rules[(rules.confidence > 0.5)&(rules.zhangs_metric > 0.5)].sort_values(by='lift',ascending=False)
vals=[len(list(rules_sub.iloc[k,:].consequents)) == 1 for k in range(rules_sub.shape[0])]
rules_sub.loc[vals,:].consequents.drop_duplicates()

424        frozenset({sector_Salud y Protección Social})
425    frozenset({modalidad_contratacion_Contratación...
205                   frozenset({duracion_contrato_q_3})
299                   frozenset({duracion_contrato_q_0})
335                     frozenset({valor_contratoo_q_0})
366          frozenset({dias_desde_primer_contrato_q_3})
204          frozenset({dias_desde_primer_contrato_q_0})
206                     frozenset({valor_contratoo_q_3})
427                  frozenset({dias_espera_inicio_q_0})
Name: consequents, dtype: object

In [196]:
vals=[list(rules_sub.iloc[k,:].consequents) == ['sector_Salud y Protección Social'] for k in range(rules_sub.shape[0])]
l=0
print(rules_sub.loc[vals,:].iloc[l,:].antecedents)
rules_sub.loc[vals,:].iloc[l,:]

frozenset({'modalidad_contratacion_Contratación régimen especial'})


antecedents           frozenset({modalidad_contratacion_Contratación...
consequents               frozenset({sector_Salud y Protección Social})
antecedent support                                             0.145898
consequent support                                             0.146244
support                                                        0.106957
confidence                                                     0.733096
lift                                                           5.012816
representativity                                                    1.0
leverage                                                       0.085621
conviction                                                     3.198738
zhangs_metric                                                  0.937255
jaccard                                                         0.57757
certainty                                                      0.687377
kulczynski                                                     0

In [197]:
vals=[list(rules_sub.iloc[k,:].consequents) == ['duracion_contrato_q_0'] for k in range(rules_sub.shape[0])]
l=0
print(rules_sub.loc[vals,:].iloc[l,:].antecedents)
rules_sub.loc[vals,:].iloc[l,:]

frozenset({'tipo_contrato_Prestación de servicios', 'valor_contratoo_q_0', 'dias_desde_primer_contrato_q_3'})


antecedents           frozenset({tipo_contrato_Prestación de servici...
consequents                          frozenset({duracion_contrato_q_0})
antecedent support                                             0.100035
consequent support                                             0.242298
support                                                        0.080132
confidence                                                     0.801038
lift                                                           3.305999
representativity                                                    1.0
leverage                                                       0.055893
conviction                                                     3.808274
zhangs_metric                                                  0.775052
jaccard                                                        0.305611
certainty                                                      0.737414
kulczynski                                                     0

In [198]:
vals=[list(rules_sub.iloc[k,:].consequents) == ['valor_contratoo_q_0'] for k in range(rules_sub.shape[0])]
l=0
print(rules_sub.loc[vals,:].iloc[l,:].antecedents)
rules_sub.loc[vals,:].iloc[l,:]

frozenset({'tipo_contrato_Prestación de servicios', 'rama_Ejecutivo', 'duracion_contrato_q_0', 'modalidad_contratacion_Contratación directa'})


antecedents           frozenset({tipo_contrato_Prestación de servici...
consequents                            frozenset({valor_contratoo_q_0})
antecedent support                                              0.11215
consequent support                                             0.248875
support                                                        0.084112
confidence                                                         0.75
lift                                                           3.013561
representativity                                                    1.0
leverage                                                       0.056201
conviction                                                       3.0045
zhangs_metric                                                  0.752567
jaccard                                                         0.30375
certainty                                                      0.667166
kulczynski                                                     0

## 4.3 MINERÍA DE LA CLASE 2

### 4.3.0 SUBCONJUNTO Y PREPRACIÓN

In [211]:
df0=dfs.copy()[dfs.clase==2]
df0.head()

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q
1,agricultura,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,0,3,0,3,-0.502233,-0.314990,0.007575,0.003116,0.000818,0.132630,0.047187,-0.261014,-0.207582,0.041610,0.177906,0.090063,0.021362,0.113700,-0.266953,0.032131,0.100277,0.052154,0.042238,-0.037100,0.025141,-0.161671,-0.109760,0.206130,0.286944,-0.508839,-0.089393,0.386624,0.709633,0.170777,0.529664,-0.574655,-0.339869,0.541785,0.226969,1.292304,1.291991,-0.388060,2,1,0,3,0,3
2,Salud y Protección Social,Ejecutivo,Centralizada,Prestación de servicios,Contratación régimen especial,No,No,No,No,No,No,No Válido,No,Si,0,2,1,0,1.089181,-0.408082,0.010449,0.011622,0.011589,0.238742,-0.031481,-0.343014,-0.053873,-0.060902,-0.056533,-0.014609,0.334820,0.020835,0.121477,-0.550172,0.169496,-0.077804,-0.090625,0.054882,0.036406,0.091204,0.087001,-0.132702,0.073605,0.015176,-0.013934,0.034000,0.075565,0.016952,0.002795,0.079164,-0.010538,0.088693,-0.077400,1.541353,1.246222,-0.408194,2,2,0,2,1,0
16,Servicio Público,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,0,0,3,1,-0.442577,-0.190577,0.004161,0.002818,0.026690,0.185955,-0.043467,-0.162269,-0.052908,0.073032,0.046968,0.133886,-0.040200,-0.095047,0.021912,0.121379,0.022787,-0.088817,0.051312,0.068660,0.055401,-0.151738,0.004320,-0.078546,0.073219,0.072032,0.195943,-0.081875,-0.014780,-0.219008,-0.012906,0.208623,-0.122016,0.212520,0.030249,1.058909,1.704955,-0.433727,2,16,0,0,3,1
32,Servicio Público,Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,No Válido,No,No,0,1,2,0,-0.397731,-0.197147,0.005842,0.006077,0.016604,0.148651,-0.036331,-0.145595,-0.000919,0.052624,0.076648,0.179538,-0.013848,-0.142886,0.058698,0.088930,0.028534,-0.112758,0.075234,0.082162,0.080118,-0.156285,0.005067,-0.117989,0.065349,0.109027,0.241389,-0.095134,-0.028666,-0.236057,-0.037698,0.281472,-0.170756,0.289937,0.022665,1.537911,2.264077,-0.760939,2,32,0,1,2,0
41,Ambiente y Desarrollo Sostenible,Corporación Autónoma,Descentralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,1,1,2,1,-0.003794,-0.450302,0.011637,0.014974,0.160415,-1.318242,-0.187274,1.544792,-0.193684,0.173967,-0.088692,-0.440738,0.323590,-0.059955,0.138974,0.447464,-0.211076,-0.780021,-0.057436,0.029708,0.115563,-0.544655,0.306729,-0.285933,0.187102,0.155016,0.334765,-0.009011,-0.171667,0.082530,0.032346,0.086452,0.199511,0.209665,0.042262,1.141662,0.357170,-0.219113,2,41,1,1,2,1


In [212]:
df_onehot=pd.concat([pd.get_dummies(df0.loc[:,cuals+[cuant+'_q' for cuant in cuants]],drop_first=False),pd.get_dummies(df0.loc[:,binarias],drop_first=True)],axis=1)
df_onehot.head()

,sector_Ambiente y Desarrollo Sostenible,sector_Ciencia Tecnología,sector_Cultura,sector_Educación Nacional,sector_Hacienda y Crédito Público,sector_Inclusión Social y Reconciliación,sector_Industria,sector_Información Estadística,sector_Inteligencia Estratégica y Contrainteligencia,sector_Ley de Justicia,sector_Minas y Energía,sector_No aplica/No pertenece,sector_Planeación,sector_Presidencia de la República,sector_Relaciones Exteriores,sector_Salud y Protección Social,sector_Servicio Público,sector_Tecnologías de la Información y las Comunicaciones,sector_Trabajo,sector_Transporte,"sector_Vivienda, Ciudad y Territorio",sector_agricultura,sector_defensa,sector_deportes,sector_interior,rama_Corporación Autónoma,rama_Ejecutivo,rama_Judicial,rama_Legislativo,tipo_contrato_Acuerdo Marco de Precios,tipo_contrato_Arrendamiento de inmuebles,tipo_contrato_Arrendamiento de muebles,tipo_contrato_Comodato,tipo_contrato_Compraventa,tipo_contrato_Consultoría,tipo_contrato_Decreto 092 de 2017,tipo_contrato_Interventoría,tipo_contrato_Obra,tipo_contrato_Otro,tipo_contrato_Prestación de servicios,tipo_contrato_Seguros,tipo_contrato_Servicios financieros,tipo_contrato_Suministros,modalidad_contratacion_Concurso de méritos abierto,modalidad_contratacion_Contratación Directa (con ofertas),modalidad_contratacion_Contratación directa,modalidad_contratacion_Contratación régimen especial,modalidad_contratacion_Contratación régimen especial (con ofertas),modalidad_contratacion_Licitación Pública Acuerdo Marco de Precios,modalidad_contratacion_Licitación pública,modalidad_contratacion_Mínima cuantía,modalidad_contratacion_Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes,modalidad_contratacion_Selección Abreviada de Menor Cuantía,modalidad_contratacion_Selección abreviada subasta inversa,dias_espera_inicio_q_0,dias_espera_inicio_q_1,dias_espera_inicio_q_2,duracion_contrato_q_0,duracion_contrato_q_1,duracion_contrato_q_2,duracion_contrato_q_3,dias_desde_primer_contrato_q_0,dias_desde_primer_contrato_q_1,dias_desde_primer_contrato_q_2,dias_desde_primer_contrato_q_3,valor_contratoo_q_0,valor_contratoo_q_1,valor_contratoo_q_2,valor_contratoo_q_3,centralizada_Descentralizada,grupo_Si,pime_Si,pago_adelantado_Si,liquidacion_Si,obligacion_ambiente_Si,obligacion_postconsumo_Si,estado_bpin_Válido,prorrogado_Si,postconflicto_Si
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False
16,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,False
32,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,F

In [213]:
df_onehot.sum(axis=0)/df_onehot.shape[0]

sector_Ambiente y Desarrollo Sostenible                                               0.041365
sector_Ciencia Tecnología                                                             0.002511
sector_Cultura                                                                        0.025558
sector_Educación Nacional                                                             0.056434
sector_Hacienda y Crédito Público                                                     0.012557
sector_Inclusión Social y Reconciliación                                              0.023785
sector_Industria                                                                      0.009307
sector_Información Estadística                                                        0.030728
sector_Inteligencia Estratégica y Contrainteligencia                                  0.000295
sector_Ley de Justicia                                                                0.015512
sector_Minas y Energía                            

In [214]:
cumplen=(df_onehot.sum(axis=0)/df_onehot.shape[0]>0.1) & (df_onehot.sum(axis=0)/df_onehot.shape[0]<0.9)
df_onehot=df_onehot.loc[:,cumplen]
df_onehot.head()

,sector_No aplica/No pertenece,sector_Salud y Protección Social,sector_Servicio Público,rama_Corporación Autónoma,rama_Ejecutivo,tipo_contrato_Prestación de servicios,modalidad_contratacion_Contratación directa,modalidad_contratacion_Contratación régimen especial,dias_espera_inicio_q_0,dias_espera_inicio_q_1,dias_espera_inicio_q_2,duracion_contrato_q_0,duracion_contrato_q_1,duracion_contrato_q_2,duracion_contrato_q_3,dias_desde_primer_contrato_q_0,dias_desde_primer_contrato_q_1,dias_desde_primer_contrato_q_2,dias_desde_primer_contrato_q_3,valor_contratoo_q_0,valor_contratoo_q_1,valor_contratoo_q_2,valor_contratoo_q_3,centralizada_Descentralizada,pime_Si,liquidacion_Si,estado_bpin_Válido,prorrogado_Si
1,False,False,False,False,True,True,True,False,True,False,False,False,False,False,True,True,False,False,False,False,False,False,True,False,False,False,True,False
2,False,True,False,False,True,True,False,True,True,False,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True
16,False,False,True,False,True,True,True,False,True,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False
32,False,False,True,False,True,True,True,False,True,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False
41,False,False,False,True,False,True,True,False,False,True,False,False,True,False,False,False,False,True,False,False,True,False,False,True,False,False,True,False


### 4.3.1 MINERÍA DE ITEMSETS FRECUENTES

In [215]:
freqitems = apriori(df_onehot, min_support=0.08, use_colnames=True)
freqitems.sort_values('support',ascending=True)

,support,itemsets
155,0.080071,"frozenset({centralizada_Descentralizada, estad..."
39,0.080219,"frozenset({sector_Servicio Público, dias_esper..."
127,0.080514,"frozenset({dias_desde_primer_contrato_q_1, dia..."
384,0.080662,"frozenset({rama_Ejecutivo, valor_contratoo_q_0..."
210,0.080662,"frozenset({dias_espera_inicio_q_1, rama_Ejecut..."
214,0.081253,"frozenset({dias_espera_inicio_q_1, rama_Ejecut..."
115,0.081253,"frozenset({dias_espera_inicio_q_1, duracion_co..."
401,0.081401,"frozenset({rama_Ejecutivo, valor_contratoo_q_3..."
160,0.081548,frozenset({tipo_contrato_Prestación de servici...
279,0.081548,frozenset({tipo_contrato_Prestación de servici...


In [216]:
freqitems =fpgrowth(df_onehot, min_support=0.08, use_colnames=True)
freqitems.sort_values('support',ascending=True).head(50)

,support,itemsets
392,0.080071,"frozenset({centralizada_Descentralizada, estad..."
275,0.080219,"frozenset({sector_Servicio Público, dias_esper..."
169,0.080514,"frozenset({dias_desde_primer_contrato_q_1, dia..."
344,0.080662,"frozenset({dias_espera_inicio_q_1, rama_Ejecut..."
214,0.080662,"frozenset({rama_Ejecutivo, valor_contratoo_q_0..."
247,0.081253,"frozenset({dias_espera_inicio_q_1, rama_Ejecut..."
301,0.081253,"frozenset({dias_espera_inicio_q_1, duracion_co..."
130,0.081401,"frozenset({rama_Ejecutivo, valor_contratoo_q_3..."
270,0.081548,frozenset({tipo_contrato_Prestación de servici...
316,0.081548,"frozenset({rama_Ejecutivo, modalidad_contratac..."


In [217]:
freqitems.shape

(454, 2)

### 4.3.2 MINERÍA DE REGLAS DE ASOCIACIÓN

In [218]:
rules = association_rules(freqitems, metric="lift", min_threshold=1.5)

rules.loc[:, "pearson"] = rules["leverage"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"] *
    (1 - rules["antecedent support"]) * (1 - rules["consequent support"])
)

rules.loc[:, "cosine"] = rules["support"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"]
)

rules.loc[:, "jaccard"] = rules["support"] / (
    rules["antecedent support"] + rules["consequent support"] - rules["support"]
)

rules.loc[:, "kuczynski"] = (rules["support"] / 2) * (
    1 / rules["antecedent support"] + 1 / rules["consequent support"]
)

rules.loc[:, "certainty"] = (
    rules["confidence"] - rules["consequent support"]
) / (1 - rules["consequent support"])

rules.head(50)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,pearson,cosine,kuczynski
0,frozenset({valor_contratoo_q_3}),frozenset({dias_desde_primer_contrato_q_0}),0.256020,0.245827,0.101640,0.396999,1.614957,1.0,0.038703,1.250701,0.511826,0.253968,0.200448,0.405230,0.205959,0.405147,0.405230
1,frozenset({dias_desde_primer_contrato_q_0}),frozenset({valor_contratoo_q_3}),0.245827,0.256020,0.101640,0.413462,1.614957,1.0,0.038703,1.268425,0.504909,0.253968,0.211621,0.405230,0.205959,0.405147,0.405230
2,frozenset({tipo_contrato_Prestación de servici...,frozenset({dias_desde_primer_contrato_q_0}),0.181858,0.245827,0.093219,0.512591,2.085175,1.0,0.048513,1.547313,0.636105,0.278710,0.353718,0.445899,0.292100,0.440883,0.445899
3,frozenset({tipo_contrato_Prestación de servici...,frozenset({valor_contratoo_q_3}),0.228099,0.256020,0.093219,0.408679,1.596276,1.0,0.034821,1.258165,0.483925,0.238473,0.205192,0.386394,0.190145,0.385750,0.386394
4,frozenset({valor_contratoo_q_3}),frozenset({tipo_contrato_Prestación de servici...,0.256020,0.228099,0.093219,0.364108,1.596276,1.0,0.034821,1.213888,0.502086,0.238473,0.176201,0.386394,0.190145,0.385750,0.386394
5,frozenset({dias_desde_primer_contrato_q_0}),frozenset({tipo_contrato_Prestación de servici...,0.245827,0.181858,0.093219,0.379207,2.085175,1.0,0.048513,1.317897,0.690059,0.278710,0.241215,0.445899,0.292100,0.440883,0.445899
6,"frozenset({rama_Ejecutivo, valor_contratoo_q_3})",frozenset({dias_desde_primer_contrato_q_0}),0.210519,0.245827,0.088935,0.422456,1.718513,1.0,0.037184,1.305829,0.529590,0.242059,0.234203,0.392117,0.211830,0.390942,0.392117
7,"frozenset({rama_Ejecutivo, dias_desde_primer_c...",frozenset({valor_contratoo_q_3}),0.205200,0.256020,0.088935,0.433405,1.692857,1.0,0.036399,1.313073,0.514950,0.238889,0.238427,0.390390,0.206519,0.388013,0.390390
8,frozenset({valor_contratoo_q_3}),"frozenset({rama_Ejecutivo, dias_desde_primer_c...",0.256020,0.205200,0.088935,0.347374,1.692857,1.0,0.036399,1.217850,0.550126,0.238889,0.178881,0.390390,0.206519,0.388013,0.390390
9,frozenset({dias_desde_primer_contrato_q_0}),"frozenset({rama_Ejecutivo, valor_contratoo_q_3})",0.245827,0.210519,0.088935,0.361779,1.718513,1.0,0.037184,1.237003,0.554384,0.242059,0.191595,0.392117,0.211830,0.390942,0.392117


In [219]:
rules_sub=rules[(rules.confidence > 0.5)&(rules.zhangs_metric > 0.5)].sort_values(by='lift',ascending=False)
rules_sub.head(50)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,pearson,cosine,kuczynski
244,frozenset({modalidad_contratacion_Contratación...,frozenset({sector_Salud y Protección Social}),0.152755,0.151573,0.112129,0.734043,4.842821,1.0,0.088975,3.190084,0.936576,0.583397,0.686529,0.736904,0.689681,0.736899,0.736904
245,frozenset({sector_Salud y Protección Social}),frozenset({modalidad_contratacion_Contratación...,0.151573,0.152755,0.112129,0.739766,4.842821,1.0,0.088975,3.255705,0.935271,0.583397,0.692847,0.736904,0.689681,0.736899,0.736904
163,"frozenset({valor_contratoo_q_3, dias_desde_pri...",frozenset({tipo_contrato_Prestación de servici...,0.101640,0.208598,0.083173,0.818314,3.922923,1.0,0.061971,4.355876,0.829387,0.366298,0.770425,0.608520,0.504755,0.571211,0.608520
164,"frozenset({duracion_contrato_q_3, dias_desde_p...",frozenset({tipo_contrato_Prestación de servici...,0.125572,0.181858,0.083173,0.662353,3.642134,1.0,0.060337,2.423067,0.829612,0.370883,0.587300,0.559852,0.472055,0.550389,0.559852
157,frozenset({tipo_contrato_Prestación de servici...,frozenset({duracion_contrato_q_3}),0.093219,0.245383,0.083173,0.892235,3.636084,1.0,0.060299,7.002398,0.799508,0.325622,0.857192,0.615593,0.481970,0.549932,0.615593
111,"frozenset({valor_contratoo_q_3, dias_desde_pri...",frozenset({duracion_contrato_q_3}),0.101640,0.245383,0.087458,0.860465,3.506616,1.0,0.062517,5.408086,0.795699,0.336938,0.815092,0.608438,0.480788,0.553787,0.608438
73,frozenset({tipo_contrato_Prestación de servici...,"frozenset({valor_contratoo_q_3, modalidad_cont...",0.174767,0.159699,0.093219,0.533390,3.339977,1.0,0.065309,1.800863,0.848969,0.386405,0.444711,0.558554,0.469447,0.557987,0.558554
82,"frozenset({valor_contratoo_q_3, modalidad_cont...",frozenset({tipo_contrato_Prestación de servici...,0.159699,0.174767,0.093219,0.583719,3.339977,1.0,0.065309,1.982392,0.833745,0.386405,0.495559,0.558554,0.469447,0.557987,0.558554
76,"frozenset({rama_Ejecutivo, valor_contratoo_q_3...",frozenset({tipo_contrato_Prestación de servici...,0.134732,0.208598,0.093219,0.691886,3.316839,1.0,0.065114,2.568536,0.807274,0.372711,0.610673,0.569385,0.469367,0.556051,0.569385
231,"frozenset({duracion_contrato_q_0, modalidad_co...",frozenset({tipo_contrato_Prestación de servici...,0.138277,0.204757,0.093810,0.678419,3.313288,1.0,0.065497,2.472915,0.810220,0.376408,0.595619,0.568286,0.470209,0.557512,0.568286


In [220]:
vals=[len(list(rules_sub.iloc[k,:].consequents)) == 1 for k in range(rules_sub.shape[0])]

In [221]:
np.array(vals).sum()

np.int64(88)

In [222]:
rules_sub.loc[vals,:].consequents.drop_duplicates()

244        frozenset({sector_Salud y Protección Social})
245    frozenset({modalidad_contratacion_Contratación...
157                   frozenset({duracion_contrato_q_3})
185                   frozenset({duracion_contrato_q_0})
227                     frozenset({valor_contratoo_q_0})
112                     frozenset({valor_contratoo_q_3})
156          frozenset({dias_desde_primer_contrato_q_0})
266          frozenset({dias_desde_primer_contrato_q_3})
378                   frozenset({duracion_contrato_q_2})
243                  frozenset({dias_espera_inicio_q_0})
Name: consequents, dtype: object

In [223]:
vals=[list(rules_sub.iloc[k,:].consequents) == ['duracion_contrato_q_0'] for k in range(rules_sub.shape[0])]
print(rules_sub.loc[vals,:].iloc[0,:].antecedents)
rules_sub.loc[vals,:].iloc[0,:]

frozenset({'valor_contratoo_q_0', 'dias_desde_primer_contrato_q_3'})


antecedents           frozenset({valor_contratoo_q_0, dias_desde_pri...
consequents                          frozenset({duracion_contrato_q_0})
antecedent support                                             0.115527
consequent support                                             0.241395
support                                                        0.086423
confidence                                                     0.748082
lift                                                              3.099
representativity                                                    1.0
leverage                                                       0.058536
conviction                                                     3.011317
zhangs_metric                                                  0.765784
jaccard                                                        0.319498
certainty                                                      0.667919
kulczynski                                                     0

## 4.4 MINERÍA DE LA CLASE 3

### 4.4.0 SUBCONJUNTO Y PREPRACIÓN

In [227]:
df0=dfs.copy()[dfs.clase==3]
df0.head()

,sector,rama,centralizada,tipo_contrato,modalidad_contratacion,grupo,pime,pago_adelantado,liquidacion,obligacion_ambiente,obligacion_postconsumo,estado_bpin,postconflicto,prorrogado,dias_espera_inicio,duracion_contrato,dias_desde_primer_contrato,valor_contratoo,MCA1,MCA2,MCA3,MCA4,MCA5,MCA6,MCA7,MCA8,MCA9,MCA10,MCA11,MCA12,MCA13,MCA14,MCA15,MCA16,MCA17,MCA18,MCA19,MCA20,MCA21,MCA22,MCA23,MCA24,MCA25,MCA26,MCA27,MCA28,MCA29,MCA30,MCA31,MCA32,MCA33,MCA34,MCA35,SE1,SE2,SE3,clase,indice,dias_espera_inicio_q,duracion_contrato_q,dias_desde_primer_contrato_q,valor_contratoo_q
14,Ambiente y Desarrollo Sostenible,Corporación Autónoma,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,No Válido,No,No,2,3,0,3,0.129805,-0.442842,0.015263,0.019625,0.146688,-1.407366,-0.168962,1.599216,-0.108490,0.129548,-0.055144,-0.402230,0.349272,-0.102312,0.164043,0.400653,-0.203536,-0.788251,-0.030020,0.027894,0.135418,-0.532851,0.307781,-0.292646,0.164645,0.167310,0.335567,-0.013272,-0.175789,0.115208,0.013700,0.110319,0.165059,0.248470,0.032868,1.466957,-1.270253,0.604457,3,14,2,3,0,3
19,Transporte,Ejecutivo,Centralizada,Obra,Contratación directa,No,No,No,Si,Si,No,Válido,No,Si,2,3,3,3,-0.111183,0.726818,-0.054001,-0.055899,0.466056,0.196426,-0.135981,0.303224,0.255181,1.401556,-0.130319,-1.397941,-0.716311,0.111667,0.385838,-1.118042,-0.334421,0.311420,-0.344544,0.023287,0.011611,0.157863,0.172594,0.109773,0.270292,-0.240390,-0.152532,-0.078040,0.104727,-0.299779,0.057525,0.007253,-0.260541,-0.031714,0.360554,1.602621,-1.940352,0.630937,3,19,2,3,3,3
29,"Vivienda, Ciudad y Territorio",Ejecutivo,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,Válido,No,No,2,0,1,0,-0.281986,-0.294440,0.011551,0.006757,0.119426,-0.063049,-0.048614,0.011249,-0.059607,0.066095,0.240529,0.273505,-0.096110,-0.195214,-0.139677,-0.174309,0.082977,-0.305554,0.081492,0.179094,-0.083223,0.020008,0.546665,2.563628,-1.323832,-0.128420,0.356478,0.068013,-0.719459,-0.025714,0.052414,0.073487,-0.296640,-0.100556,0.201403,1.806525,-1.962812,0.793963,3,29,2,0,1,0
31,Educación Nacional,Corporación Autónoma,Centralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,No Válido,No,No,0,0,2,0,0.079415,-0.432563,0.020150,0.022008,0.013452,-1.047666,-0.110473,0.896411,0.145191,-0.040289,-0.167094,0.798808,-0.381324,-0.628761,-0.426660,-0.397626,-0.028509,1.039252,0.075334,-0.180215,-0.134792,0.249601,-0.407485,-0.121404,0.111527,-0.229936,0.063644,0.324101,-0.012738,0.076958,-0.153960,-0.063286,-0.016394,0.002442,0.108144,1.601453,-1.946982,0.637009,3,31,0,0,2,0
36,Salud y Protección Social,Ejecutivo,Descentralizada,Prestación de servicios,Contratación directa,No,No,No,No,No,No,No Válido,No,Si,0,2,0,1,0.477179,-0.371925,0.008764,0.010483,-0.003859,0.196321,-0.064273,-0.258153,-0.067814,-0.073885,-0.056396,-0.026628,0.310411,-0.001188,0.159446,-0.523528,0.192738,-0.092248,-0.094312,0.076097,0.064411,0.119318,0.043365,-0.207043,0.054160,0.060926,0.001486,0.056117,0.070264,0.015607,-0.037954,0.103874,-0.018967,0.111269,-0.091549,1.664626,-0.993948,0.376302,3,36,0,2,0,1


In [228]:
df_onehot=pd.concat([pd.get_dummies(df0.loc[:,cuals+[cuant+'_q' for cuant in cuants]],drop_first=False),pd.get_dummies(df0.loc[:,binarias],drop_first=True)],axis=1)
df_onehot.head()

,sector_Ambiente y Desarrollo Sostenible,sector_Ciencia Tecnología,sector_Cultura,sector_Educación Nacional,sector_Hacienda y Crédito Público,sector_Inclusión Social y Reconciliación,sector_Industria,sector_Información Estadística,sector_Inteligencia Estratégica y Contrainteligencia,sector_Ley de Justicia,sector_Minas y Energía,sector_No aplica/No pertenece,sector_Planeación,sector_Presidencia de la República,sector_Relaciones Exteriores,sector_Salud y Protección Social,sector_Servicio Público,sector_Tecnologías de la Información y las Comunicaciones,sector_Trabajo,sector_Transporte,"sector_Vivienda, Ciudad y Territorio",sector_agricultura,sector_defensa,sector_deportes,sector_interior,rama_Corporación Autónoma,rama_Ejecutivo,rama_Judicial,rama_Legislativo,tipo_contrato_Acuerdo Marco de Precios,tipo_contrato_Arrendamiento de inmuebles,tipo_contrato_Arrendamiento de muebles,tipo_contrato_Asociación Público Privada,tipo_contrato_Comisión,tipo_contrato_Comodato,tipo_contrato_Compraventa,tipo_contrato_Consultoría,tipo_contrato_Decreto 092 de 2017,tipo_contrato_Interventoría,tipo_contrato_Obra,tipo_contrato_Operaciones de Crédito Público,tipo_contrato_Otro,tipo_contrato_Prestación de servicios,tipo_contrato_Seguros,tipo_contrato_Servicios financieros,tipo_contrato_Suministros,modalidad_contratacion_Concurso de méritos abierto,modalidad_contratacion_Contratación Directa (con ofertas),modalidad_contratacion_Contratación directa,modalidad_contratacion_Contratación régimen especial,modalidad_contratacion_Contratación régimen especial (con ofertas),modalidad_contratacion_Licitación Pública Acuerdo Marco de Precios,modalidad_contratacion_Licitación pública,modalidad_contratacion_Licitación pública Obra Publica,modalidad_contratacion_Mínima cuantía,modalidad_contratacion_Selección Abreviada de Menor Cuantía,modalidad_contratacion_Selección abreviada subasta inversa,dias_espera_inicio_q_0,dias_espera_inicio_q_1,dias_espera_inicio_q_2,duracion_contrato_q_0,duracion_contrato_q_1,duracion_contrato_q_2,duracion_contrato_q_3,dias_desde_primer_contrato_q_0,dias_desde_primer_contrato_q_1,dias_desde_primer_contrato_q_2,dias_desde_primer_contrato_q_3,valor_contratoo_q_0,valor_contratoo_q_1,valor_contratoo_q_2,valor_contratoo_q_3,centralizada_Descentralizada,grupo_Si,pime_Si,pago_adelantado_Si,liquidacion_Si,obligacion_ambiente_Si,estado_bpin_Válido,prorrogado_Si,postconflicto_Si
14,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
19,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,True,True,True,True,False
29,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False
31,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False

In [229]:
df_onehot.sum(axis=0)/df_onehot.shape[0]

sector_Ambiente y Desarrollo Sostenible                               0.036963
sector_Ciencia Tecnología                                             0.003009
sector_Cultura                                                        0.027650
sector_Educación Nacional                                             0.052579
sector_Hacienda y Crédito Público                                     0.009599
sector_Inclusión Social y Reconciliación                              0.028080
sector_Industria                                                      0.009742
sector_Información Estadística                                        0.027937
sector_Inteligencia Estratégica y Contrainteligencia                  0.000143
sector_Ley de Justicia                                                0.014040
sector_Minas y Energía                                                0.007163
sector_No aplica/No pertenece                                         0.160888
sector_Planeación                                   

In [230]:
cumplen=(df_onehot.sum(axis=0)/df_onehot.shape[0]>0.1) & (df_onehot.sum(axis=0)/df_onehot.shape[0]<0.9)
df_onehot=df_onehot.loc[:,cumplen]
df_onehot.head()

,sector_No aplica/No pertenece,sector_Salud y Protección Social,sector_Servicio Público,rama_Corporación Autónoma,rama_Ejecutivo,tipo_contrato_Prestación de servicios,modalidad_contratacion_Contratación directa,modalidad_contratacion_Contratación régimen especial,dias_espera_inicio_q_0,dias_espera_inicio_q_1,dias_espera_inicio_q_2,duracion_contrato_q_0,duracion_contrato_q_1,duracion_contrato_q_2,duracion_contrato_q_3,dias_desde_primer_contrato_q_0,dias_desde_primer_contrato_q_1,dias_desde_primer_contrato_q_2,dias_desde_primer_contrato_q_3,valor_contratoo_q_0,valor_contratoo_q_1,valor_contratoo_q_2,valor_contratoo_q_3,centralizada_Descentralizada,pime_Si,liquidacion_Si,estado_bpin_Válido,prorrogado_Si
14,False,False,False,True,False,True,True,False,False,False,True,False,False,False,True,True,False,False,False,False,False,False,True,False,False,False,False,False
19,False,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,True,True,True
29,False,False,False,False,True,True,True,False,False,False,True,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False
31,False,False,False,True,False,True,True,False,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False
36,False,True,False,False,True,True,True,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,True,False,False,False,True


### 4.4.1 MINERÍA DE ITEMSETS FRECUENTES

In [231]:
freqitems = apriori(df_onehot, min_support=0.08, use_colnames=True)
freqitems.sort_values('support',ascending=True)

,support,itemsets
277,0.080372,frozenset({tipo_contrato_Prestación de servici...
429,0.080659,frozenset({tipo_contrato_Prestación de servici...
128,0.080659,"frozenset({dias_espera_inicio_q_2, duracion_co..."
111,0.080659,"frozenset({dias_espera_inicio_q_0, dias_desde_..."
140,0.081089,"frozenset({valor_contratoo_q_1, duracion_contr..."
353,0.081232,frozenset({tipo_contrato_Prestación de servici...
145,0.081232,"frozenset({valor_contratoo_q_2, duracion_contr..."
165,0.081232,"frozenset({dias_espera_inicio_q_0, modalidad_c..."
315,0.081232,"frozenset({valor_contratoo_q_3, modalidad_cont..."
85,0.081519,frozenset({tipo_contrato_Prestación de servici...


In [232]:
freqitems =fpgrowth(df_onehot, min_support=0.08, use_colnames=True)
freqitems.sort_values('support',ascending=True).head(300)

,support,itemsets
438,0.080372,frozenset({tipo_contrato_Prestación de servici...
48,0.080659,"frozenset({dias_espera_inicio_q_2, duracion_co..."
152,0.080659,"frozenset({dias_espera_inicio_q_0, dias_desde_..."
119,0.080659,frozenset({tipo_contrato_Prestación de servici...
311,0.081089,"frozenset({valor_contratoo_q_1, duracion_contr..."
117,0.081232,"frozenset({valor_contratoo_q_3, modalidad_cont..."
361,0.081232,"frozenset({valor_contratoo_q_2, duracion_contr..."
390,0.081232,"frozenset({dias_espera_inicio_q_0, modalidad_c..."
175,0.081232,frozenset({tipo_contrato_Prestación de servici...
243,0.081519,frozenset({tipo_contrato_Prestación de servici...


In [233]:
freqitems.shape

(451, 2)

### 4.1.2 MINERÍA DE REGLAS DE ASOCIACIÓN

In [240]:
rules = association_rules(freqitems, metric="lift", min_threshold=1.5)

rules.loc[:, "pearson"] = rules["leverage"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"] *
    (1 - rules["antecedent support"]) * (1 - rules["consequent support"])
)

rules.loc[:, "cosine"] = rules["support"] / np.sqrt(
    rules["antecedent support"] * rules["consequent support"]
)

rules.loc[:, "jaccard"] = rules["support"] / (
    rules["antecedent support"] + rules["consequent support"] - rules["support"]
)

rules.loc[:, "kuczynski"] = (rules["support"] / 2) * (
    1 / rules["antecedent support"] + 1 / rules["consequent support"]
)

rules.loc[:, "certainty"] = (
    rules["confidence"] - rules["consequent support"]
) / (1 - rules["consequent support"])

rules.head(50)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,pearson,cosine,kuczynski
0,frozenset({valor_contratoo_q_3}),frozenset({duracion_contrato_q_3}),0.253152,0.256017,0.149284,0.589700,2.303361,1.0,0.084472,1.813266,0.757653,0.414809,0.448509,0.586400,0.445136,0.586391,0.586400
1,frozenset({duracion_contrato_q_3}),frozenset({valor_contratoo_q_3}),0.256017,0.253152,0.149284,0.583100,2.303361,1.0,0.084472,1.791433,0.760571,0.414809,0.441788,0.586400,0.445136,0.586391,0.586400
2,frozenset({tipo_contrato_Prestación de servici...,frozenset({duracion_contrato_q_3}),0.175931,0.256017,0.124355,0.706840,2.760910,1.0,0.079314,2.537808,0.773965,0.404285,0.605959,0.596285,0.477287,0.585947,0.596285
3,frozenset({tipo_contrato_Prestación de servici...,frozenset({valor_contratoo_q_3}),0.216619,0.253152,0.124355,0.574074,2.267706,1.0,0.069518,1.753470,0.713606,0.360017,0.429702,0.532651,0.388110,0.531038,0.532651
4,frozenset({valor_contratoo_q_3}),frozenset({tipo_contrato_Prestación de servici...,0.253152,0.216619,0.124355,0.491228,2.267706,1.0,0.069518,1.539749,0.748513,0.360017,0.350544,0.532651,0.388110,0.531038,0.532651
5,frozenset({duracion_contrato_q_3}),frozenset({tipo_contrato_Prestación de servici...,0.256017,0.175931,0.124355,0.485730,2.760910,1.0,0.079314,1.602406,0.857279,0.404285,0.375938,0.596285,0.477287,0.585947,0.596285
6,"frozenset({valor_contratoo_q_3, modalidad_cont...",frozenset({duracion_contrato_q_3}),0.159599,0.256017,0.116619,0.730700,2.854106,1.0,0.075759,2.762656,0.772997,0.390034,0.638030,0.593106,0.473979,0.576925,0.593106
7,frozenset({modalidad_contratacion_Contratación...,frozenset({valor_contratoo_q_3}),0.200000,0.253152,0.116619,0.583095,2.303339,1.0,0.065989,1.791409,0.707310,0.346530,0.441780,0.521881,0.379404,0.518279,0.521881
8,frozenset({valor_contratoo_q_3}),frozenset({modalidad_contratacion_Contratación...,0.253152,0.200000,0.116619,0.460668,2.303339,1.0,0.065989,1.483316,0.757648,0.346530,0.325835,0.521881,0.379404,0.518279,0.521881
9,frozenset({duracion_contrato_q_3}),"frozenset({valor_contratoo_q_3, modalidad_cont...",0.256017,0.159599,0.116619,0.455512,2.854106,1.0,0.075759,1.543471,0.873176,0.390034,0.352109,0.593106,0.473979,0.576925,0.593106


In [241]:
rules_sub=rules[(rules.confidence > 0.5)&(rules.zhangs_metric > 0.5)].sort_values(by='lift',ascending=False)
rules_sub.head(50)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,pearson,cosine,kuczynski
380,frozenset({modalidad_contratacion_Contratación...,"frozenset({dias_espera_inicio_q_0, sector_Salu...",0.155444,0.091691,0.081232,0.522581,5.699395,1.0,0.066979,1.902540,0.976303,0.489637,0.474387,0.704259,0.640561,0.680422,0.704259
377,"frozenset({dias_espera_inicio_q_0, sector_Salu...",frozenset({modalidad_contratacion_Contratación...,0.091691,0.155444,0.081232,0.885937,5.699395,1.0,0.066979,7.404325,0.907777,0.489637,0.864944,0.704259,0.640561,0.680422,0.704259
376,"frozenset({dias_espera_inicio_q_0, modalidad_c...",frozenset({sector_Salud y Protección Social}),0.097851,0.156447,0.081232,0.830161,5.306341,1.0,0.065924,4.966782,0.899570,0.469371,0.798662,0.674696,0.610772,0.656540,0.674696
381,frozenset({sector_Salud y Protección Social}),"frozenset({dias_espera_inicio_q_0, modalidad_c...",0.156447,0.097851,0.081232,0.519231,5.306341,1.0,0.065924,1.876470,0.962057,0.469371,0.467084,0.674696,0.610772,0.656540,0.674696
372,frozenset({modalidad_contratacion_Contratación...,frozenset({sector_Salud y Protección Social}),0.155444,0.156447,0.116189,0.747465,4.777755,1.0,0.091870,3.340347,0.936228,0.593704,0.700630,0.745070,0.697966,0.745066,0.745070
373,frozenset({sector_Salud y Protección Social}),frozenset({modalidad_contratacion_Contratación...,0.156447,0.155444,0.116189,0.742674,4.777755,1.0,0.091870,3.282046,0.937341,0.593704,0.695312,0.745070,0.697966,0.745066,0.745070
147,"frozenset({valor_contratoo_q_3, dias_desde_pri...",frozenset({tipo_contrato_Prestación de servici...,0.097851,0.216619,0.080659,0.824305,3.805321,1.0,0.059463,4.458744,0.817171,0.344975,0.775722,0.598330,0.485834,0.554016,0.598330
148,"frozenset({duracion_contrato_q_3, dias_desde_p...",frozenset({tipo_contrato_Prestación de servici...,0.128510,0.175931,0.080659,0.627648,3.567574,1.0,0.058050,2.213143,0.825824,0.360435,0.548154,0.543058,0.455565,0.536430,0.543058
141,frozenset({tipo_contrato_Prestación de servici...,frozenset({duracion_contrato_q_3}),0.090688,0.256017,0.080659,0.889415,3.474046,1.0,0.057441,6.727730,0.783176,0.303177,0.851361,0.602234,0.458330,0.529352,0.602234
239,"frozenset({duracion_contrato_q_0, modalidad_co...","frozenset({rama_Ejecutivo, tipo_contrato_Prest...",0.144699,0.178223,0.086963,0.600990,3.372115,1.0,0.061174,2.059539,0.822459,0.368549,0.514454,0.544466,0.454375,0.541524,0.544466


In [236]:
vals=[len(list(rules_sub.iloc[k,:].consequents)) == 1 for k in range(rules_sub.shape[0])]

In [237]:
np.array(vals).sum()

np.int64(88)

In [238]:
rules_sub.loc[vals,:].consequents.drop_duplicates()

377    frozenset({modalidad_contratacion_Contratación...
376        frozenset({sector_Salud y Protección Social})
141                   frozenset({duracion_contrato_q_3})
267                   frozenset({duracion_contrato_q_0})
227                     frozenset({valor_contratoo_q_0})
250          frozenset({dias_desde_primer_contrato_q_3})
140          frozenset({dias_desde_primer_contrato_q_0})
142                     frozenset({valor_contratoo_q_3})
378                  frozenset({dias_espera_inicio_q_0})
Name: consequents, dtype: object

In [239]:
vals=[list(rules_sub.iloc[k,:].consequents) == ['duracion_contrato_q_0'] for k in range(rules_sub.shape[0])]
print(rules_sub.loc[vals,:].iloc[0,:].antecedents)
rules_sub.loc[vals,:].iloc[0,:]

frozenset({'valor_contratoo_q_0', 'dias_desde_primer_contrato_q_3'})


antecedents           frozenset({valor_contratoo_q_0, dias_desde_pri...
consequents                          frozenset({duracion_contrato_q_0})
antecedent support                                             0.113467
consequent support                                             0.243553
support                                                        0.086103
confidence                                                     0.758838
lift                                                           3.115701
representativity                                                    1.0
leverage                                                       0.058468
conviction                                                     3.136681
zhangs_metric                                                  0.765956
jaccard                                                        0.317821
certainty                                                      0.681192
kulczynski                                                     0